To run this, press "*Runtime*" and press "*Run all*" on a Google Colab L4 instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

In [ ]:
import os
import tarfile
import shutil
import pandas as pd
import torch

from datetime import datetime
from datasets import Dataset, concatenate_datasets

/home/vml/notebooks/test/med-lt-project/colab-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

import os

BASE_DIR = "/home/vml/notebooks/test/med-lt-project"

TAR_DIR = f"{BASE_DIR}/tar"
CSV_DIR = f"{BASE_DIR}/csv"

LOCAL_IMAGE_ROOT = f"{BASE_DIR}/temp_images"

GEMMA_DIR = f"{BASE_DIR}/gemma4"

CHECKPOINT_ROOT = f"{GEMMA_DIR}/checkpoints"
EXPERIMENT_ROOT = f"{GEMMA_DIR}/experiments"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

EXPERIMENT_NAME = f"fin-trained_vqarad_{timestamp}"

EXPERIMENT_DIR = f"{EXPERIMENT_ROOT}/{EXPERIMENT_NAME}"
CHECKPOINT_DIR = f"{CHECKPOINT_ROOT}/{EXPERIMENT_NAME}"
PREDICTIONS_DIR = f"{EXPERIMENT_DIR}/predictions"
BEST_MODEL_DIR = f"{EXPERIMENT_DIR}/best_model"

os.makedirs(LOCAL_IMAGE_ROOT, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("LOCAL_IMAGE_ROOT:", LOCAL_IMAGE_ROOT)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("PREDICTIONS_DIR:", PREDICTIONS_DIR)
print("BEST_MODEL_DIR:", BEST_MODEL_DIR)

BASE_DIR: /home/vml/notebooks/test/med-lt-project
LOCAL_IMAGE_ROOT: /home/vml/notebooks/test/med-lt-project/temp_images
CHECKPOINT_DIR: /home/vml/notebooks/test/med-lt-project/gemma4/checkpoints/fin-trained_vqarad_20260520_115902
PREDICTIONS_DIR: /home/vml/notebooks/test/med-lt-project/gemma4/experiments/fin-trained_vqarad_20260520_115902/predictions
BEST_MODEL_DIR: /home/vml/notebooks/test/med-lt-project/gemma4/experiments/fin-trained_vqarad_20260520_115902/best_model


In [ ]:
DATASETS = {

    "pathvqa": {
        "csv_train": "path-vqa-train.csv",
        "csv_val": "path-vqa-validate.csv",
        "csv_test": "path-vqa-test.csv",
        "tar_files": [
            "pathvqa.tar",
        ],
    },

    "vqarad": {
        "csv_train": "vqa-rad-train.csv",
        "csv_val": "vqa-rad-validate.csv",
        "csv_test": "vqa-rad-test.csv",
        "tar_files": [
            "vqarad.tar",
        ],
    },

    "slake": {
        "csv_train": "slake-train.csv",
        "csv_val": "slake-validate.csv",
        "csv_test": "slake-test.csv",
        "tar_files": [
            "slake.tar",
        ],
    },
}

In [ ]:
TRAIN_DATASETS = [
    "vqarad",
    # "vqarad",
    # "slake",
]

VAL_DATASETS = [
    "vqarad",
]

TEST_DATASETS = [
    "vqarad",
]

In [ ]:
GLOBAL_RULES = """
Atsakyk trumpai lietuvių kalba.
Pateik tik medicininį atsakymą.
Nenaudok paaiškinimų.
"""

In [ ]:

ALL_REQUIRED_DATASETS = sorted(
    set(
        TRAIN_DATASETS +
        VAL_DATASETS +
        TEST_DATASETS
    )
)

print("Required datasets:", ALL_REQUIRED_DATASETS)

for dataset_name in ALL_REQUIRED_DATASETS:

    cfg = DATASETS[dataset_name]

    dataset_tar_dir = os.path.join(
        TAR_DIR,
        dataset_name,
    )

    print("\nDATASET:", dataset_name)

    for tar_name in cfg["tar_files"]:

        tar_path = os.path.join(
            dataset_tar_dir,
            tar_name,
        )

        if not os.path.exists(tar_path):

            print("MISSING TAR:", tar_path)
            continue

        print("EXTRACTING:", tar_path)

        with tarfile.open(tar_path, "r") as tar:
            tar.extractall(LOCAL_IMAGE_ROOT)

print("\nExtraction complete.")

Required datasets: ['vqarad']

DATASET: vqarad
EXTRACTING: /home/vml/notebooks/test/med-lt-project/tar/vqarad/vqarad.tar

Extraction complete.


In [ ]:


def convert_to_conversation(sample):

    image_path = os.path.join(
        LOCAL_IMAGE_ROOT,
        str(sample["dataset"]),
        str(sample["image"]),
    )

    question = str(
        sample["question_lt"]
    )

    answer = str(
        sample["answer_lt"]
    )

    conversation = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text":
f"""
{GLOBAL_RULES}

Klausimas:
{question}
"""
                },
                {
                    "type": "image",
                    "image": image_path,
                },
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": answer,
                }
            ],
        },
    ]

    return {
        "messages": conversation
    }

In [ ]:


def load_medical_split(dataset_names, split, max_samples=None):

    converted_datasets = []

    for dataset_name in dataset_names:

        cfg = DATASETS[dataset_name]

        csv_name = cfg[f"csv_{split}"]

        csv_path = os.path.join(
            CSV_DIR,
            dataset_name,
            csv_name,
        )

        print("\nLOADING CSV:", csv_path)

        df = pd.read_csv(
            csv_path,
            encoding="utf-8-sig",
        )

        if max_samples is not None:
            df = df.sample(
            n=min(max_samples, len(df)),
            random_state=3407,
            )


        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
        )

        required_columns = [
            "image",
            "question_lt",
            "answer_lt",
        ]

        for col in required_columns:

            if col not in df.columns:

                raise ValueError(
                    f"Missing column '{col}' in {csv_path}. Found columns: {list(df.columns)}"
                )

        df["dataset"] = dataset_name

        raw_dataset = Dataset.from_pandas(
            df,
            preserve_index=False,
        )

        converted = [
            convert_to_conversation(sample)
            for sample in raw_dataset
        ]

        converted_dataset = Dataset.from_list(
            converted
        )

        converted_datasets.append(
            converted_dataset
        )

    if len(converted_datasets) == 1:
        return converted_datasets[0]

    return concatenate_datasets(
        converted_datasets
    )

In [ ]:
train_dataset = load_medical_split(
    TRAIN_DATASETS,
    "train",
)

val_dataset = load_medical_split(
    VAL_DATASETS,
    "val",
    max_samples=None,
)

test_dataset = load_medical_split(
    TEST_DATASETS,
    "test",
)

print("TRAIN:", train_dataset)
print("VAL:", val_dataset)
print("TEST:", test_dataset)

print("\nExample:")
print(train_dataset[0])


LOADING CSV: /home/vml/notebooks/test/med-lt-project/csv/vqarad/vqa-rad-train.csv

LOADING CSV: /home/vml/notebooks/test/med-lt-project/csv/vqarad/vqa-rad-validate.csv

LOADING CSV: /home/vml/notebooks/test/med-lt-project/csv/vqarad/vqa-rad-test.csv
TRAIN: Dataset({
    features: ['messages'],
    num_rows: 1793
})
VAL: Dataset({
    features: ['messages'],
    num_rows: 225
})
TEST: Dataset({
    features: ['messages'],
    num_rows: 226
})

Example:
{'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': '\n\nAtsakyk trumpai lietuvių kalba.\nPateik tik medicininį atsakymą.\nNenaudok paaiškinimų.\n\n\nKlausimas:\nAr smegenų sritys infarktas?\n', 'image': None}, {'type': 'image', 'text': None, 'image': '/home/vml/notebooks/test/med-lt-project/temp_images/vqarad/output_train_images_img_0.jpg'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': 'taip', 'image': None}]}]}


In [ ]:


def verify_images(dataset, name, limit_print=20):

    missing = []

    for sample in dataset:

        image_path = sample["messages"][0]["content"][1]["image"]

        if not os.path.exists(image_path):
            missing.append(image_path)

    print(f"{name} missing images:", len(missing))

    for path in missing[:limit_print]:
        print(path)

    if len(missing) > 0:
        raise FileNotFoundError(
            f"{name} has {len(missing)} missing images."
        )

verify_images(train_dataset, "train_dataset")
verify_images(val_dataset, "val_dataset")
verify_images(test_dataset, "test_dataset")

train_dataset missing images: 0
val_dataset missing images: 0
test_dataset missing images: 0


### Unsloth

In [ ]:
import os

os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

import shutil
import torch
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

torch._dynamo.config.suppress_errors = True
torch._dynamo.config.recompile_limit = 128
torch._dynamo.config.cache_size_limit = 128
torch._dynamo.config.accumulated_cache_size_limit = 512

cache_dir = "/home/vml/notebooks/test/med-lt-project/unsloth_compiled_cache"

if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("Deleted:", cache_dir)

from unsloth import FastVisionModel



In [ ]:
from unsloth import FastVisionModel
import torch

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E4B-it",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

model = model.to("cuda")

==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.697 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 2130/2130 [00:28<00:00, 75.88it/s]


We now add LoRA adapters for parameter efficient fine-tuning, allowing us to train only 1% of all model parameters efficiently.

**[NEW]** We also support fine-tuning only the vision component, only the language component, or both. Additionally, you can choose to fine-tune the attention modules, the MLP layers, or both!

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,                           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,                  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,               # We support rank stabilized LoRA
    loftq_config = None,               # And LoftQ
    target_modules = "all-linear",    # Optional now! Can specify a list if needed
)

In [ ]:
from unsloth import get_chat_template

processor = get_chat_template(
    processor,
    "gemma-4"
)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This notebook needs an NVIDIA GPU.")

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")
print("Model device:", next(model.parameters()).device)


GPU = NVIDIA GB10. Max memory = 119.697 GB.
10.406 GB of memory reserved.
Model device: cuda:0


In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    processing_class=processor.tokenizer,

    data_collator=UnslothVisionDataCollator(
        model,
        processor,
    ),

    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,

        max_grad_norm=0.3,
        warmup_ratio=0.03,

        num_train_epochs=2,

        learning_rate=2e-4,

        logging_steps=20,

        eval_strategy="steps",
        eval_steps=25,

        save_strategy="steps",
        save_steps=25,

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",

        seed=3407,

        output_dir=CHECKPOINT_DIR,
        report_to="tensorboard",

        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={
            "skip_prepare_dataset": True,
        },
        max_length=1024,
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,793 | Num Epochs = 2 | Total steps = 898
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,222,144 of 8,037,378,592 (0.51% trained)


Step,Training Loss,Validation Loss
25,1.402778,5.648582
50,1.172167,5.399175
75,1.001740,5.330225
100,0.829950,5.279557
125,0.740325,5.308118
150,0.741494,5.118680
175,0.686730,5.069569
200,0.700391,5.042856
225,0.632555,5.010806
250,0.585133,4.992093


Unsloth: Restored added_tokens_decoder metadata in /home/vml/notebooks/test/med-lt-project/gemma4/checkpoints/fin-trained_vqarad_20260520_115902/checkpoint-25/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/vml/notebooks/test/med-lt-project/gemma4/checkpoints/fin-trained_vqarad_20260520_115902/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/vml/notebooks/test/med-lt-project/gemma4/checkpoints/fin-trained_vqarad_20260520_115902/checkpoint-75/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/vml/notebooks/test/med-lt-project/gemma4/checkpoints/fin-trained_vqarad_20260520_115902/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/vml/notebooks/test/med-lt-project/gemma4/checkpoints/fin-trained_vqarad_20260520_115902/checkpoint-125/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/vml/notebooks/test/med-lt-proje

KeyboardInterrupt: 

In [ ]:
model.save_pretrained(BEST_MODEL_DIR)
processor.save_pretrained(BEST_MODEL_DIR)

print("Saved LoRA model to:")
print(BEST_MODEL_DIR)
print(f"Saved model eval loss: {trainer.state.best_metric}")

Unsloth: Restored added_tokens_decoder metadata in /home/vml/notebooks/test/med-lt-project/gemma4/experiments/fin-trained_vqarad_20260520_115902/best_model/tokenizer_config.json.


Saved LoRA model to:
/home/vml/notebooks/test/med-lt-project/gemma4/experiments/fin-trained_vqarad_20260520_115902/best_model
Saved model eval loss: 4.775665283203125


In [ ]:
from unsloth import FastVisionModel
import torch
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

BEST_DIR = "/home/vml/notebooks/test/med-lt-project/gemma4/experiments/fin-trained_pathvqa_20260519_160544/best_model"

model, processor = FastVisionModel.from_pretrained(
    BEST_DIR,
    load_in_4bit=True,
)

FastVisionModel.for_inference(model)

model = model.to("cuda")
model.eval()

device = next(model.parameters()).device

print("Loaded best model from:")
print(BEST_DIR)
print("Model device:", device)


==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.697 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 2130/2130 [00:53<00:00, 39.84it/s]


Loaded best model from:
/home/vml/notebooks/test/med-lt-project/gemma4/experiments/fin-trained_pathvqa_20260519_160544/best_model
Model device: cuda:0


In [ ]:
import pandas as pd
from tqdm import tqdm
import torch
import gc

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

results = []

FastVisionModel.for_inference(model)

model = model.to("cuda")
model.eval()

device = next(model.parameters()).device


def clean_prediction(text):
    text = str(text).strip()

    for marker in ["user", "model", "assistant"]:
        if marker in text:
            text = text.split(marker)[-1].strip()

    if "Klausimas:" in text:
        text = text.split("Klausimas:")[-1].strip()

    lines = [line.strip() for line in text.split("\n") if line.strip()]
    if len(lines) > 0:
        text = lines[0]

    text = text.strip()
    text = text.strip(" .")

    return text


test_range = range(len(test_dataset))


with torch.inference_mode():
    for idx in tqdm(test_range):

        sample = test_dataset[idx]

        messages = sample["messages"]

        image = messages[0]["content"][1]["image"]
        question = messages[0]["content"][0]["text"]
        ground_truth = messages[1]["content"][0]["text"]

        inference_messages = [
            messages[0]
        ]

        input_text = processor.apply_chat_template(
            inference_messages,
            add_generation_prompt=True,
        )

        inputs = processor(
            image,
            input_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).to(device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            use_cache=True,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
        )

        generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]

        decoded = processor.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        )

        predicted_answer = clean_prediction(decoded)

        results.append({
            "image": image,
            "question_lt": question,
            "correct_answer_lt": ground_truth,
            "predicted_answer_lt": predicted_answer,
        })

        if (idx + 1) % 100 == 0:
            partial_df = pd.DataFrame(results)
            partial_df.to_csv(
                f"{EXPERIMENT_DIR}/test_predictions_partial.csv",
                index=False,
            )

results_df = pd.DataFrame(results)

results_df.to_csv(
    f"{EXPERIMENT_DIR}/test_predictions.csv",
    index=False,
)

results_df.head()

100%|███████████████████████████████████████████████████████████████████████████████████████████████| 226/226 [02:25<00:00,  1.55it/s]


,image,question_lt,correct_answer_lt,predicted_answer_lt
0,/home/vml/notebooks/test/med-lt-project/temp_i...,\n\nAtsakyk trumpai lietuvių kalba.\nPateik ti...,Ne,Ne
1,/home/vml/notebooks/test/med-lt-project/temp_i...,\n\nAtsakyk trumpai lietuvių kalba.\nPateik ti...,taip,Ne
2,/home/vml/notebooks/test/med-lt-project/temp_i...,\n\nAtsakyk trumpai lietuvių kalba.\nPateik ti...,plaučių mazgeliai,diffuzinis parenchiminis infiltratas
3,/home/vml/notebooks/test/med-lt-project/temp_i...,\n\nAtsakyk trumpai lietuvių kalba.\nPateik ti...,Padidintas,"karnės, kraukštai ir bendras šlapimo/p"
4,/home/vml/notebooks/test/med-lt-project/temp_i...,\n\nAtsakyk trumpai lietuvių kalba.\nPateik ti...,hidrocefalija,pleurocelys


In [ ]:
results_df = pd.DataFrame(results)

RESULT_PATH = f"{PREDICTIONS_DIR}/test_predictions.csv"

results_df.to_csv(
    RESULT_PATH,
    index = False,
)

print("Saved predictions:")
print(RESULT_PATH)

results_df.head()

Saved predictions:
/home/vml/notebooks/test/med-lt-project/gemma4/experiments/fin-trained_vqarad_20260520_115902/predictions/test_predictions.csv


,image,question_lt,correct_answer_lt,predicted_answer_lt
0,/home/vml/notebooks/test/med-lt-project/temp_i...,\n\nAtsakyk trumpai lietuvių kalba.\nPateik ti...,Ne,Ne
1,/home/vml/notebooks/test/med-lt-project/temp_i...,\n\nAtsakyk trumpai lietuvių kalba.\nPateik ti...,taip,Ne
2,/home/vml/notebooks/test/med-lt-project/temp_i...,\n\nAtsakyk trumpai lietuvių kalba.\nPateik ti...,plaučių mazgeliai,diffuzinis parenchiminis infiltratas
3,/home/vml/notebooks/test/med-lt-project/temp_i...,\n\nAtsakyk trumpai lietuvių kalba.\nPateik ti...,Padidintas,"karnės, kraukštai ir bendras šlapimo/p"
4,/home/vml/notebooks/test/med-lt-project/temp_i...,\n\nAtsakyk trumpai lietuvių kalba.\nPateik ti...,hidrocefalija,pleurocelys


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).